# 6. The 2027 fare calendar

Three designs, all with the same rules (one price per day for everyone, only ever cheaper than today's fare, discount from 0 to 30% by expected pressure, displayed in the five levels of the team's mockup):

* **year** - city pressure ranked within the year: discounts follow the seasons;
* **month** - the same pressure ranked within each month: every month has cheap and dear days;
* **hotspots** - visitors where visitors concentrate, ranked within the month.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from prisa import store, figures
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
con = store.connect()
from prisa import pricing, fares, reference, schemas

In [ ]:
daily = store.read('daily_fares')
for design in pricing.DESIGNS:
    figures.calendar_heatmap(daily[daily['calendar_id'] == pricing.calendar_ids(design)[0]], f'2027 - {design} design')

In [ ]:
summary = store.read('fare_calendar_2027_monthly')
summary[summary['calendar_id'] == 'prisa-2027-month'].round(3)

## Fairness checks

No product ever costs more than its published price, and resident titles (T-usual, T-jove) never cost more per trip than the T-usual reference - by construction, since flat passes don't follow the daily price. The pipeline refuses to write a calendar that breaks either rule; here they are run again explicitly.

In [ ]:
products = reference.load(schemas.FARE_PRODUCTS)
for design in pricing.DESIGNS:
    cal = daily[daily['calendar_id'] == pricing.calendar_ids(design)[0]]
    prices = fares.product_prices(cal, products)
    print(design, 'above published:', len(fares.never_above_published(prices)),
          '| resident above reference:', len(fares.check_resident_fairness(prices, 22.80, 40, ['t_usual', 't_jove'])))

In [ ]:
# the deliverable: calendar_2027.csv for the chosen design
chosen = daily[daily['calendar_id'] == 'prisa-2027-month'][['day', 'expected_pressure', 'tier', 'discount']]
single = products.set_index('product_id').loc['single', 'price_eur']
chosen = chosen.assign(single_ticket_eur=(single * (1 - chosen['discount'])).round(2))
out = store.OUTPUTS / 'calendar_2027.csv'; out.parent.mkdir(parents=True, exist_ok=True)
chosen.to_csv(out, index=False); out